In [ ]:
%pip install yfinance

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import os

print("1. Locating Local Sri Lankan Data...")
file_name = 'CBSL_Forex_Raw.csv'

# Smart path finder (works whether your notebook is in the root or a subfolder)
if os.path.exists(f'../data/raw/{file_name}'):
    cbsl_file_path = f'../data/raw/{file_name}'
    out_dir = '../data/raw'
elif os.path.exists(f'data/raw/{file_name}'):
    cbsl_file_path = f'data/raw/{file_name}'
    out_dir = 'data/raw'
elif os.path.exists(file_name):
    cbsl_file_path = file_name
    out_dir = '.'
else:
    raise FileNotFoundError(f"Could not find {file_name}. Please ensure it is saved in the data/raw folder and does not have a hidden double extension (like .csv.csv).")

print(f"File found at: {cbsl_file_path}")
df = pd.read_csv(cbsl_file_path)

# Filter out the headers and find only the Exchange Rate rows
df = df[df['Item Name'].astype(str).str.startswith('TT Rates -')]

# Drop unneeded CBSL metadata columns
cols_to_drop = [c for c in ['Unnamed: 0', 'Unit', 'Scale'] if c in df.columns]
df = df.drop(columns=cols_to_drop)

# Melt the dataset from "Wide" to "Long"
df_long = df.melt(id_vars=['Item Name'], var_name='Date', value_name='Rate')

# Extract "Buying" vs "Selling" and the exact Currency Name
df_long['Type'] = df_long['Item Name'].apply(lambda x: 'Buying' if 'Buying' in str(x) else 'Selling')

def get_curr(item):
    item = str(item)
    if 'USD' in item: return 'USD'
    if 'GBP' in item: return 'GBP'
    if 'EURO' in item: return 'EUR'
    if 'CAD' in item: return 'CAD'
    if 'AUD' in item: return 'AUD'
    if 'JPY' in item: return 'JPY'
    return 'UNKNOWN'

df_long['Currency'] = df_long['Item Name'].apply(get_curr)

# Clean and drop NaNs (which represent closed banks on weekends/holidays)
df_long['Rate'] = pd.to_numeric(df_long['Rate'], errors='coerce')
df_long = df_long.dropna(subset=['Rate'])

# Pivot so 'Buying' and 'Selling' are side-by-side
df_pivot = df_long.pivot_table(index=['Date', 'Currency'], columns='Type', values='Rate').reset_index()

print("2. Calculating the 'Gold Standard' Mid-Rate...")
df_pivot['LKR_Rate'] = (df_pivot['Buying'] + df_pivot['Selling']) / 2
df_pivot['Date'] = pd.to_datetime(df_pivot['Date'], errors='coerce')
df_pivot = df_pivot.dropna(subset=['Date']).sort_values(['Currency', 'Date'])

print("3. Forward-Filling missing weekends & holidays...")
final_rows = []
for curr in df_pivot['Currency'].unique():
    curr_df = df_pivot[df_pivot['Currency'] == curr].set_index('Date')
    full_idx = pd.date_range(start=curr_df.index.min(), end=curr_df.index.max(), freq='D')
    curr_df = curr_df.reindex(full_idx)
    curr_df['Currency'] = curr
    curr_df['LKR_Rate'] = curr_df['LKR_Rate'].ffill() 
    curr_df = curr_df.reset_index().rename(columns={'index': 'Date'})
    final_rows.append(curr_df[['Date', 'Currency', 'LKR_Rate']])

cbsl_clean_df = pd.concat(final_rows, ignore_index=True)

print("4. Fetching Global Benchmark (USD Index) via yfinance...")
dxy = yf.download('DX-Y.NYB', start=cbsl_clean_df['Date'].min(), end=cbsl_clean_df['Date'].max())
if isinstance(dxy.columns, pd.MultiIndex):
    dxy = dxy['Close'].reset_index()
else:
    dxy = dxy.reset_index()[['Date', 'Close']]

dxy.columns = ['Date', 'USD_Index']
dxy['Date'] = pd.to_datetime(dxy['Date']).dt.tz_localize(None)

print("5. Merging Local Targets with Global Benchmark...")
final_dataset = pd.merge(cbsl_clean_df, dxy, on='Date', how='left')
final_dataset['USD_Index'] = final_dataset.groupby('Currency')['USD_Index'].ffill()
final_dataset = final_dataset.dropna().reset_index(drop=True)

# Save the final file exactly where Notebook 2 expects it
output_path = os.path.join(out_dir, 'LKR_Forex_Macro_Raw.csv')
final_dataset.to_csv(output_path, index=False)

print(f"SUCCESS: Master data saved to {output_path}")
print(f"Total Rows: {len(final_dataset)}")
print(f"Currencies: {final_dataset['Currency'].unique()}")

1. Locating Local Sri Lankan Data...
File found at: ../data/raw/CBSL_Forex_Raw.csv
2. Calculating the 'Gold Standard' Mid-Rate...
3. Forward-Filling missing weekends & holidays...
4. Fetching Global Benchmark (USD Index) via yfinance...


[*********************100%***********************]  1 of 1 completed


5. Merging Local Targets with Global Benchmark...
SUCCESS: Master data saved to ../data/raw\LKR_Forex_Macro_Raw.csv
Total Rows: 35346
Currencies: ['AUD' 'CAD' 'EUR' 'GBP' 'JPY' 'USD']
